# ARC-v0.10 — Boundary-Aware Selective Fidelity Mitigation

This post-confirmatory notebook tests whether the v0.9 boundary model can be converted into an
actionable retrieval policy.

Policies:

- Always-PQ32: PQ32 search → PQ32 feedback
- Always-SQ8: PQ32 search → SQ8 feedback
- Random selective SQ8: budget-matched random allocation
- Boundary-aware selective SQ8: v0.9 fit-only risk model allocates SQ8 feedback

Frozen high-fidelity budgets: **10%, 25%, 50%**.

The v0.9 fit split is used to reconstruct the risk model and choose thresholds.
The v0.9 validation split is used only for mitigation evaluation.

Primary system question:

\[
\text{recovery}(b)=
\frac{U_{selective,b}-U_{PQ32}}
{U_{SQ8}-U_{PQ32}}
\]

A useful policy should beat Always-PQ32 and budget-matched random allocation while using far fewer
SQ8 feedback searches than Always-SQ8.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, gc, sqlite3, time
import numpy as np
import pandas as pd
import faiss
import psutil
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("FAISS:", faiss.__version__)
print("RAM GB:", psutil.virtual_memory().total / 1024**3)


## 1. Paths and provenance


In [ ]:
SEED = 20260816
DIM = 384
N_DOCS = 5_233_329
NPROBE = 64
TOP_RETRIEVE = 100
MAX_ROUNDS = 4

BUDGETS = [0.10, 0.25, 0.50]
PRIMARY_BUDGET = 0.25
BOOTSTRAP_SAMPLES = 10_000

FROZEN_CONFIGS = [
    {"method": "mean", "k": 20, "alpha": 0.3, "temperature": None},
    {"method": "softmax", "k": 5, "alpha": 0.5, "temperature": 0.1},
]

ROOT = Path("/content/drive/MyDrive/hc-rars-external-confirmation-hotpotqa-5m-v1")
ARC_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0")
INDEX_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/hotpotqa-rebuilt-v3")

SHARD_ROOT = ROOT / "stage1/corpus-embedding-shards-v3"
SHARD_MANIFEST = SHARD_ROOT / "manifest.json"
QUERY_EMB = ROOT / "stage1/query_embeddings.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
CORPUS_DB = ROOT / "stage1/corpus_ids.sqlite"
DEV_QRELS = ROOT / "source/hotpotqa/qrels/dev.tsv"

PQ32_PATH = INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
SQ8_PATH = INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfsq8-nlist4096-seed20260816.faiss"

V08_RUNS = sorted(
    [p for p in (ARC_ROOT / "sealed-hotpotqa-h1-h4-confirmation-v08").glob("*")
     if (p / "report.json").is_file()],
    key=lambda p: p.stat().st_mtime,
)
V09_RUNS = sorted(
    [p for p in (ARC_ROOT / "boundary-stability-map-v09").glob("*")
     if (p / "report.json").is_file()],
    key=lambda p: p.stat().st_mtime,
)

if not V08_RUNS:
    raise FileNotFoundError("ARC-v0.8 report not found.")
if not V09_RUNS:
    raise FileNotFoundError("ARC-v0.9 report not found.")

V08_RUN = V08_RUNS[-1]
V09_RUN = V09_RUNS[-1]

OUT_ROOT = ARC_ROOT / "boundary-aware-selective-fidelity-v010"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT.mkdir(parents=True, exist_ok=False)

print("v0.8:", V08_RUN)
print("v0.9:", V09_RUN)
print("v0.10:", OUT)


## 2. Helpers and DEV alignment


In [ ]:
def sha256_file(path, chunk_size=64*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

with open(SHARD_MANIFEST, "r", encoding="utf-8") as f:
    shard_manifest = json.load(f)

assert shard_manifest["status"] == "CORPUS_EMBEDDING_SHARDS_COMPLETE"
shards = sorted(shard_manifest["shards"], key=lambda x: int(x["shard_id"]))
SHARD_ENDS = np.asarray([int(s["end_row"]) for s in shards], dtype=np.int64)

queries = np.load(QUERY_EMB, mmap_mode="r")
with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]
with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x) for x in split["dev_query_ids"]]
query_row = {qid:i for i,qid in enumerate(query_ids)}
dev_query_rows = np.asarray([query_row[q] for q in dev_ids], dtype=np.int64)

Q_DEV = np.asarray(queries[dev_query_rows], dtype=np.float32)
Q_DEV /= np.maximum(np.linalg.norm(Q_DEV, axis=1, keepdims=True), 1e-12)

dev = pd.read_csv(DEV_QRELS, sep="\t")
dev["query-id"] = dev["query-id"].astype(str)
dev["corpus-id"] = dev["corpus-id"].astype(str)

unique_doc_ids = dev["corpus-id"].drop_duplicates().tolist()

with sqlite3.connect(str(CORPUS_DB)) as con:
    con.execute("CREATE TEMP TABLE requested_ids (doc_id TEXT PRIMARY KEY)")
    con.executemany("INSERT INTO requested_ids(doc_id) VALUES (?)", [(x,) for x in unique_doc_ids])
    mapped = pd.read_sql_query(
        "SELECT r.doc_id, d.row_id FROM requested_ids r LEFT JOIN documents d ON d.doc_id=r.doc_id",
        con,
    )

assert mapped["row_id"].notna().all()
mapped["row_id"] = mapped["row_id"].astype(np.int64)
dev = dev.merge(mapped, left_on="corpus-id", right_on="doc_id", how="left", validate="many_to_one")
dev = dev.rename(columns={"row_id":"corpus-row"})
dev["corpus-row"] = dev["corpus-row"].astype(np.int64)

dev_qrels = {
    str(qid): set(g.loc[g["score"]>0, "corpus-row"].astype(np.int64).tolist())
    for qid,g in dev.groupby("query-id")
}

print("DEV alignment — PASS")


In [ ]:
def load_rows_from_shards(rows):
    rows = np.asarray(rows, dtype=np.int64)
    shape = rows.shape
    flat = rows.reshape(-1)
    unique_rows, inverse = np.unique(flat, return_inverse=True)
    shard_ids = np.searchsorted(SHARD_ENDS, unique_rows, side="right")

    vectors = np.empty((len(unique_rows), DIM), dtype=np.float32)

    for sid in np.unique(shard_ids):
        mask = shard_ids == sid
        start = int(shards[int(sid)]["start_row"])
        local = unique_rows[mask] - start
        arr = np.load(SHARD_ROOT / shards[int(sid)]["file"], mmap_mode="r")
        vectors[mask] = np.asarray(arr[local], dtype=np.float32)

    return vectors[inverse].reshape(*shape, DIM)

def normalize_rows(x):
    x = np.asarray(x, np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

def feedback_matrix(ids, scores, cfg):
    k = int(cfg["k"])
    x = load_rows_from_shards(np.asarray(ids[:, :k], np.int64))

    if cfg["method"] == "mean":
        f = x.mean(axis=1)
    else:
        z = np.asarray(scores[:, :k], np.float64) / float(cfg["temperature"])
        z -= z.max(axis=1, keepdims=True)
        w = np.exp(np.clip(z, -60, 60))
        w /= np.maximum(w.sum(axis=1, keepdims=True), 1e-12)
        f = (x * w[:,:,None]).sum(axis=1)

    return normalize_rows(f)

def anchored_update(q0, f, alpha):
    return normalize_rows((1.0-float(alpha))*q0 + float(alpha)*f)

def ndcg_at_10(qids, ids):
    discounts = 1.0 / np.log2(np.arange(2,12))
    out = np.zeros(len(qids), np.float32)

    for i,qid in enumerate(qids):
        rel = dev_qrels[str(qid)]
        hits = np.asarray([1.0 if int(d) in rel else 0.0 for d in ids[i,:10]], np.float32)
        dcg = float((hits*discounts).sum())
        ideal = min(len(rel),10)
        idcg = float(discounts[:ideal].sum())
        out[i] = dcg/idcg if idcg else 0.0

    return out

def jaccard_distance_rows(a,b):
    out = np.empty(len(a), np.float32)
    for i in range(len(a)):
        A,B = set(map(int,a[i])), set(map(int,b[i]))
        out[i] = 1.0 - len(A&B)/max(len(A|B),1)
    return out

def score_entropy(scores,k=20):
    z = np.asarray(scores[:,:k], np.float64)
    z -= z.max(axis=1, keepdims=True)
    p = np.exp(np.clip(z,-60,60))
    p /= np.maximum(p.sum(axis=1, keepdims=True),1e-12)
    return -np.sum(p*np.log(np.maximum(p,1e-12)), axis=1)

def score_margin(scores):
    return np.asarray(scores[:,0]-scores[:,9], np.float32)

def cfg_key(cfg):
    temp = "none" if cfg["temperature"] is None else str(cfg["temperature"]).replace(".","p")
    return f"{cfg['method']}-k{cfg['k']}-a{str(cfg['alpha']).replace('.','p')}-t{temp}"


## 3. Reconstruct v0.9 fit/validation split and risk model


In [ ]:
split_path = V09_RUN / "boundary_query_split.csv"
if not split_path.is_file():
    raise FileNotFoundError(split_path)

boundary_split = pd.read_csv(split_path, dtype={"query_id":str})
fit_ids = set(boundary_split.loc[boundary_split["split"]=="fit","query_id"])
val_ids = set(boundary_split.loc[boundary_split["split"]=="validation","query_id"])

id_to_pos = {qid:i for i,qid in enumerate(dev_ids)}
fit_positions = np.asarray([id_to_pos[q] for q in dev_ids if q in fit_ids], dtype=np.int64)
val_positions = np.asarray([id_to_pos[q] for q in dev_ids if q in val_ids], dtype=np.int64)
val_qids = [dev_ids[i] for i in val_positions]

print("fit:",len(fit_positions))
print("validation:",len(val_positions))


In [ ]:
pq32 = faiss.read_index(str(PQ32_PATH))
sq8 = faiss.read_index(str(SQ8_PATH))
pq32.nprobe = NPROBE
sq8.nprobe = NPROBE

pq_scores0,pq_ids0 = pq32.search(np.ascontiguousarray(Q_DEV,np.float32), TOP_RETRIEVE)
sq_scores0,sq_ids0 = sq8.search(np.ascontiguousarray(Q_DEV,np.float32), TOP_RETRIEVE)

pq_ndcg0 = ndcg_at_10(dev_ids,pq_ids0)
sq_ndcg0 = ndcg_at_10(dev_ids,sq_ids0)

features_df = pd.DataFrame({
    "query_id":dev_ids,
    "initial_candidate_divergence":jaccard_distance_rows(pq_ids0,sq_ids0),
    "initial_abs_utility_gap":np.abs(sq_ndcg0-pq_ndcg0),
    "pq32_entropy20":score_entropy(pq_scores0,20),
    "sq8_entropy20":score_entropy(sq_scores0,20),
    "pq32_margin1_10":score_margin(pq_scores0),
    "sq8_margin1_10":score_margin(sq_scores0),
})
features_df["entropy_gap"] = np.abs(features_df["pq32_entropy20"]-features_df["sq8_entropy20"])
features_df["margin_gap"] = np.abs(features_df["pq32_margin1_10"]-features_df["sq8_margin1_10"])

print("Initial features — READY")


In [ ]:
fit_files = sorted(V09_RUN.glob("fit-*.parquet"))
if not fit_files:
    raise FileNotFoundError("No v0.9 fit-*.parquet artifacts found.")

fit_boundary = pd.concat([pd.read_parquet(p) for p in fit_files], ignore_index=True)

rows = []
for keys,g in fit_boundary.groupby(
    ["query_id","method","alpha","k","temperature","config_key"],
    dropna=False,
):
    qid,method,alpha,k,temp,cfgk = keys
    g = g.sort_values("iteration")
    slope = float(np.polyfit(
        g["iteration"].to_numpy(np.float64),
        g["abs_utility_gap"].to_numpy(np.float64),
        1,
    )[0])

    rows.append({
        "query_id":str(qid),
        "method":method,
        "alpha":float(alpha),
        "k":int(k),
        "temperature":temp,
        "config_key":cfgk,
        "H3_slope":slope,
    })

fit_slopes = pd.DataFrame(rows)
fit_model_df = fit_slopes.merge(features_df,on="query_id",how="left",validate="many_to_one")
fit_model_df["is_softmax"] = (fit_model_df["method"]=="softmax").astype(float)
fit_model_df["temperature_numeric"] = pd.to_numeric(
    fit_model_df["temperature"],errors="coerce"
).fillna(1.0)

FEATURES = [
    "initial_candidate_divergence",
    "initial_abs_utility_gap",
    "pq32_entropy20",
    "sq8_entropy20",
    "pq32_margin1_10",
    "sq8_margin1_10",
    "entropy_gap",
    "margin_gap",
    "alpha",
    "k",
    "is_softmax",
    "temperature_numeric",
]

X_fit = fit_model_df[FEATURES].to_numpy(np.float64)
y_fit = fit_model_df["H3_slope"].to_numpy(np.float64)

high_threshold = float(np.quantile(y_fit,0.75))
y_fit_high = (y_fit >= high_threshold).astype(int)

risk_model = Pipeline([
    ("scale",StandardScaler()),
    ("model",LogisticRegression(C=0.5,max_iter=2000,class_weight="balanced")),
])
risk_model.fit(X_fit,y_fit_high)

fit_auc = roc_auc_score(y_fit_high,risk_model.predict_proba(X_fit)[:,1])

print("Reconstructed fit AUC:",fit_auc)
print("High-amplification threshold:",high_threshold)


## 4. Freeze fit-only selection thresholds


In [ ]:
def config_matrix(positions,cfg):
    x = features_df.iloc[positions].copy()
    x["alpha"] = float(cfg["alpha"])
    x["k"] = int(cfg["k"])
    x["is_softmax"] = float(cfg["method"]=="softmax")
    x["temperature_numeric"] = 1.0 if cfg["temperature"] is None else float(cfg["temperature"])
    return x[FEATURES].to_numpy(np.float64)

risk_thresholds = {}

for cfg in FROZEN_CONFIGS:
    ck = cfg_key(cfg)
    p_fit = risk_model.predict_proba(config_matrix(fit_positions,cfg))[:,1]
    risk_thresholds[ck] = {}

    for budget in BUDGETS:
        threshold = float(np.quantile(p_fit,1.0-budget))
        risk_thresholds[ck][str(budget)] = threshold
        print(ck,budget,threshold,float(np.mean(p_fit>=threshold)))

(OUT/"fit_only_risk_thresholds.json").write_text(
    json.dumps(risk_thresholds,indent=2),
    encoding="utf-8",
)
print("THRESHOLDS — FROZEN")


## 5. Seal mitigation protocol


In [ ]:
protocol = {
    "status":"SEALED_BEFORE_V010_VALIDATION_MITIGATION",
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "source_v08_sha256":sha256_file(V08_RUN/"report.json"),
    "source_v09_sha256":sha256_file(V09_RUN/"report.json"),
    "risk_model_training":"v0.9 fit split only",
    "budgets":BUDGETS,
    "primary_budget":PRIMARY_BUDGET,
    "feedback_configs":FROZEN_CONFIGS,
    "policies":["always_pq32","always_sq8","random_selective_sq8","boundary_selective_sq8"],
    "utility_search_retriever":"PQ32",
    "high_fidelity_feedback_retriever":"SQ8",
    "test_access_allowed":False,
}

raw = json.dumps(protocol,sort_keys=True,separators=(",",":")).encode()
protocol_sha = hashlib.sha256(raw).hexdigest()
protocol["protocol_sha256"] = protocol_sha

(OUT/"mitigation_protocol.json").write_text(json.dumps(protocol,indent=2),encoding="utf-8")
print("Protocol SHA:",protocol_sha)


## 6. Frozen validation selectors


In [ ]:
val_risk = {}

for cfg in FROZEN_CONFIGS:
    ck = cfg_key(cfg)
    val_risk[ck] = risk_model.predict_proba(config_matrix(val_positions,cfg))[:,1]

def boundary_selector(cfg,budget):
    ck = cfg_key(cfg)
    return val_risk[ck] >= risk_thresholds[ck][str(budget)]

def random_selector(cfg,budget):
    bmask = boundary_selector(cfg,budget)
    n_select = int(bmask.sum())

    ck = cfg_key(cfg)
    score = np.asarray([
        int.from_bytes(
            hashlib.sha256(f"{qid}|{ck}|{budget}|{SEED}".encode()).digest()[:8],
            "big",
        )
        for qid in val_qids
    ],dtype=np.uint64)

    order = np.argsort(score)
    mask = np.zeros(len(score),dtype=bool)
    mask[order[:n_select]] = True
    return mask

for cfg in FROZEN_CONFIGS:
    for budget in BUDGETS:
        b = boundary_selector(cfg,budget)
        r = random_selector(cfg,budget)
        assert b.sum() == r.sum()
        print(cfg_key(cfg),budget,int(b.sum()),float(b.mean()))


## 7. Validation policy simulator


In [ ]:
Q_VAL = Q_DEV[val_positions].copy()

def simulate_policy(cfg, policy, selector=None):
    q0 = Q_VAL
    q = q0.copy()

    sq8_feedback_searches = 0
    trajectory = []

    for t in range(MAX_ROUNDS+1):
        pq_scores,pq_ids = pq32.search(
            np.ascontiguousarray(q,np.float32),
            TOP_RETRIEVE,
        )

        ndcg = ndcg_at_10(val_qids,pq_ids)
        trajectory.append({
            "iteration":t,
            "mean_ndcg":float(ndcg.mean()),
        })

        if t == MAX_ROUNDS:
            return {
                "final_ndcg":ndcg.copy(),
                "trajectory":pd.DataFrame(trajectory),
                "extra_sq8_query_searches":sq8_feedback_searches,
            }

        f = feedback_matrix(pq_ids,pq_scores,cfg)

        if policy == "always_pq32":
            pass

        elif policy == "always_sq8":
            sq_scores,sq_ids = sq8.search(
                np.ascontiguousarray(q,np.float32),
                TOP_RETRIEVE,
            )
            f = feedback_matrix(sq_ids,sq_scores,cfg)
            sq8_feedback_searches += len(q)

        elif policy in {"boundary_selective_sq8","random_selective_sq8"}:
            chosen = np.flatnonzero(selector)

            if len(chosen):
                sq_scores,sq_ids = sq8.search(
                    np.ascontiguousarray(q[chosen],np.float32),
                    TOP_RETRIEVE,
                )
                f[chosen] = feedback_matrix(sq_ids,sq_scores,cfg)
                sq8_feedback_searches += len(chosen)

        else:
            raise ValueError(policy)

        q = anchored_update(q0,f,cfg["alpha"])


## 8. Run all validation policies


In [ ]:
results = {}
trajectory_frames = []

for cfg in FROZEN_CONFIGS:
    ck = cfg_key(cfg)
    print("="*80)
    print("CONFIG:",ck)

    low = simulate_policy(cfg,"always_pq32")
    high = simulate_policy(cfg,"always_sq8")

    results[(ck,"always_pq32",None)] = low
    results[(ck,"always_sq8",None)] = high

    for name,res in [("always_pq32",low),("always_sq8",high)]:
        tmp = res["trajectory"].copy()
        tmp["config_key"] = ck
        tmp["policy"] = name
        tmp["budget"] = np.nan
        trajectory_frames.append(tmp)

    for budget in BUDGETS:
        bmask = boundary_selector(cfg,budget)
        rmask = random_selector(cfg,budget)

        sel = simulate_policy(cfg,"boundary_selective_sq8",bmask)
        rnd = simulate_policy(cfg,"random_selective_sq8",rmask)

        results[(ck,"boundary_selective_sq8",budget)] = sel
        results[(ck,"random_selective_sq8",budget)] = rnd

        for name,res in [("boundary_selective_sq8",sel),("random_selective_sq8",rnd)]:
            tmp = res["trajectory"].copy()
            tmp["config_key"] = ck
            tmp["policy"] = name
            tmp["budget"] = budget
            trajectory_frames.append(tmp)

trajectory_df = pd.concat(trajectory_frames,ignore_index=True)
trajectory_df.to_csv(OUT/"policy_trajectories.csv",index=False)

print("POLICY RUNS — COMPLETE")


## 9. Paired bootstrap and cost-quality summary


In [ ]:
def bootstrap_mean(x,samples=BOOTSTRAP_SAMPLES,seed=0,chunk=250):
    x = np.asarray(x,np.float64)
    rng = np.random.default_rng(seed)
    draws = []

    for start in range(0,samples,chunk):
        b = min(chunk,samples-start)
        idx = rng.integers(0,len(x),size=(b,len(x)),dtype=np.int32)
        draws.append(x[idx].mean(axis=1))

    draws = np.concatenate(draws)

    return {
        "mean":float(x.mean()),
        "ci_low":float(np.quantile(draws,0.025)),
        "ci_high":float(np.quantile(draws,0.975)),
    }

summary = []

for ci,cfg in enumerate(FROZEN_CONFIGS):
    ck = cfg_key(cfg)

    low = results[(ck,"always_pq32",None)]["final_ndcg"]
    high = results[(ck,"always_sq8",None)]["final_ndcg"]
    denom = float((high-low).mean())

    summary.append({
        "config_key":ck,
        "policy":"always_pq32",
        "budget":0.0,
        "mean_final_ndcg":float(low.mean()),
        "extra_sq8_fraction":0.0,
        "recovery_fraction":0.0,
    })

    summary.append({
        "config_key":ck,
        "policy":"always_sq8",
        "budget":1.0,
        "mean_final_ndcg":float(high.mean()),
        "extra_sq8_fraction":1.0,
        "recovery_fraction":1.0,
    })

    for bi,budget in enumerate(BUDGETS):
        sel = results[(ck,"boundary_selective_sq8",budget)]["final_ndcg"]
        rnd = results[(ck,"random_selective_sq8",budget)]["final_ndcg"]

        s_low = bootstrap_mean(sel-low,seed=SEED+100+ci*10+bi)
        s_rnd = bootstrap_mean(sel-rnd,seed=SEED+200+ci*10+bi)
        r_low = bootstrap_mean(rnd-low,seed=SEED+300+ci*10+bi)

        actual = float(boundary_selector(cfg,budget).mean())

        summary.append({
            "config_key":ck,
            "policy":"boundary_selective_sq8",
            "budget":budget,
            "mean_final_ndcg":float(sel.mean()),
            "extra_sq8_fraction":actual,
            "delta_vs_always_pq32":s_low["mean"],
            "delta_vs_always_pq32_ci_low":s_low["ci_low"],
            "delta_vs_always_pq32_ci_high":s_low["ci_high"],
            "delta_vs_random":s_rnd["mean"],
            "delta_vs_random_ci_low":s_rnd["ci_low"],
            "delta_vs_random_ci_high":s_rnd["ci_high"],
            "recovery_fraction":float((sel-low).mean()/denom) if abs(denom)>1e-12 else np.nan,
        })

        summary.append({
            "config_key":ck,
            "policy":"random_selective_sq8",
            "budget":budget,
            "mean_final_ndcg":float(rnd.mean()),
            "extra_sq8_fraction":actual,
            "delta_vs_always_pq32":r_low["mean"],
            "delta_vs_always_pq32_ci_low":r_low["ci_low"],
            "delta_vs_always_pq32_ci_high":r_low["ci_high"],
            "recovery_fraction":float((rnd-low).mean()/denom) if abs(denom)>1e-12 else np.nan,
        })

summary_df = pd.DataFrame(summary)
display(summary_df)

summary_df.to_csv(OUT/"mitigation_summary.csv",index=False)


## 10. Primary 25% budget decision


In [ ]:
primary_df = summary_df[
    (summary_df["policy"]=="boundary_selective_sq8")
    & np.isclose(summary_df["budget"],PRIMARY_BUDGET)
].copy()

primary_df["positive_vs_low"] = primary_df["delta_vs_always_pq32_ci_low"] > 0
primary_df["positive_vs_random"] = primary_df["delta_vs_random_ci_low"] > 0

display(primary_df)
primary_df.to_csv(OUT/"primary_25pct_budget_decision.csv",index=False)


## 11. Pareto figures


In [ ]:
for ck in [cfg_key(c) for c in FROZEN_CONFIGS]:
    sub = summary_df[summary_df["config_key"]==ck].copy()

    fig,ax = plt.subplots(figsize=(7,5))

    for policy,g in sub.groupby("policy"):
        g = g.sort_values("extra_sq8_fraction")
        ax.plot(
            g["extra_sq8_fraction"],
            g["mean_final_ndcg"],
            marker="o",
            label=policy,
        )

    ax.set_xlabel("Fraction of extra SQ8 feedback searches")
    ax.set_ylabel("Final nDCG@10")
    ax.set_title(f"Selective fidelity Pareto — {ck}")
    ax.legend()
    fig.tight_layout()

    path = OUT / f"pareto-{ck}.png"
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.show()

    print("Saved:",path)


## 12. Final report


In [ ]:
report = {
    "status":"ARC_V010_BOUNDARY_AWARE_SELECTIVE_FIDELITY_COMPLETE",
    "protocol_sha256":protocol_sha,
    "source_v08_report_sha256":sha256_file(V08_RUN/"report.json"),
    "source_v09_report_sha256":sha256_file(V09_RUN/"report.json"),
    "risk_model_fit_auc":float(fit_auc),
    "validation_queries":len(val_qids),
    "budgets":BUDGETS,
    "primary_budget":PRIMARY_BUDGET,
    "feedback_configs":FROZEN_CONFIGS,
    "primary_results":primary_df.to_dict(orient="records"),
    "all_results":summary_df.to_dict(orient="records"),
    "test_accessed":False,
    "completed_at_utc":datetime.now(timezone.utc).isoformat(),
}

REPORT = OUT/"report.json"
REPORT.write_text(json.dumps(report,indent=2,default=float),encoding="utf-8")
report_sha = sha256_file(REPORT)
(OUT/"REPORT_SHA256.txt").write_text(report_sha+"\n",encoding="utf-8")

print("Saved:",OUT)
print("Report SHA:",report_sha)
display(primary_df)


## Interpretation

The strongest Full-Paper result would be:

- selective SQ8 beats Always-PQ32;
- selective SQ8 beats budget-matched random allocation;
- 25% high-fidelity budget recovers a large fraction of Always-SQ8 benefit;
- the pattern holds under both frozen feedback rules.

If the boundary-aware policy does **not** beat random allocation, retain that result. It would mean
that v0.9 has descriptive/predictive value but the static allocation policy is insufficient.

Do not retune validation thresholds after observing v0.10. A dynamic round-wise mitigation would be
a separate experiment.
